# 05 · Quality gate
Last step of the pipeline. Any failing check raises an exception, which fails the pipeline run and triggers the on-failure alert, so wrong numbers never reach the report silently.

In [ ]:
ROOT = "Files/bc_extract"

checks = {
    "Duplicate fact rows": """
        SELECT COUNT(*) FROM (
          SELECT Company, document_type, document_no, line_no FROM gold_fact_sales_line
          GROUP BY Company, document_type, document_no, line_no HAVING COUNT(*) > 1)""",

    "Customers without exactly one current version": """
        SELECT COUNT(*) FROM (
          SELECT Company, customer_no FROM gold_dim_customer WHERE customer_sk <> -1
          GROUP BY Company, customer_no
          HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1)""",

    "Days not reconciling to Business Central": """
        SELECT COUNT(*) FROM gold_recon_daily_sales WHERE ABS(difference) > 0.01""",

    # completeness: every Silver invoice / credit line must reach Gold (catches late-arriving rows being skipped)
    "Silver invoice lines missing from Gold": """
        SELECT COUNT(*) FROM silver_sales_invoice_line l
        LEFT ANTI JOIN gold_fact_sales_line f
          ON f.Company = l.Company AND f.document_type = 'INVOICE'
         AND f.document_no = l.Document_No AND f.line_no = l.Line_No""",

    "Silver credit memo lines missing from Gold": """
        SELECT COUNT(*) FROM silver_sales_cr_memo_line l
        LEFT ANTI JOIN gold_fact_sales_line f
          ON f.Company = l.Company AND f.document_type = 'CREDIT_MEMO'
         AND f.document_no = l.Document_No AND f.line_no = l.Line_No""",
}

failed = {}
for name, sql in checks.items():
    bad = spark.sql(sql).first()[0]
    print(f"{'PASS' if bad == 0 else 'FAIL'}  {name}: {bad}")
    if bad:
        failed[name] = bad

# freshness: newest folder must be loaded
latest_folder = sorted(f.name.strip("/") for f in mssparkutils.fs.ls(ROOT) if f.isDir)[-1]
latest_loaded = spark.sql("SELECT MAX(batch_id) FROM etl_batch_log").first()[0]
print(f"{'PASS' if latest_folder == latest_loaded else 'FAIL'}  Latest batch loaded: {latest_loaded} (newest folder {latest_folder})")
if latest_folder != latest_loaded:
    failed["Data is stale"] = latest_folder

# information
late = spark.sql("""
    SELECT COUNT(*) FROM silver_sales_invoice_header
    WHERE _batch_id LIKE 'incr_%' AND Posting_Date < to_date(substring(_batch_id, 6))""").first()[0]
print(f"INFO  Late-arriving invoices picked up (posted before the day they arrived): {late}")
print(f"INFO  Rows in quarantine: {spark.sql('SELECT COUNT(*) FROM silver_quarantine').first()[0]}")
missing_once = spark.sql(
    "SELECT COUNT(*) FROM silver_sales_order_status WHERE order_status = 'MISSING_ONCE'").first()[0]
print(f"INFO  Orders missing once (awaiting confirmation): {missing_once}")

if failed:
    raise Exception(f"QUALITY GATE FAILED: {failed}")
print("Quality gate passed")